<a href="https://colab.research.google.com/github/jeolin/BCCE_Experiment/blob/main/randomized_spectral_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Research on Copper Sulfate Absorbance Spectrum

Copper(II) sulfate solutions appear blue because they absorb light in the red-orange region of the visible spectrum and transmit blue-green light. The hydrated copper(II) ion, [Cu(H₂O)₆]²⁺, exhibits a characteristic broad absorption band due to d-d electronic transitions.

*   **Expected Shape**: The spectrum typically shows a broad absorption peak.
*   **Lambda Max (Absorbance Maximum)**: For aqueous copper(II) sulfate, the absorption maximum ($\lambda_{max}$) is generally found in the range of **750 nm to 820 nm**, corresponding to the lowest energy d-d transition. The exact position can vary slightly with concentration and solvent.

I will now generate a synthetic spectrum reflecting these properties.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# --- Define the fixed, broad wavelength range for underlying data ---
# This ensures that changing plot limits doesn't require re-calculating the spectrum
wavelengths_data = np.linspace(200, 1200, 1000) # Very broad range for calculation

def plot_copper_sulfate_spectrum(
    lambda_max_input=790,
    max_absorbance_input=1.0,
    min_wavelength_plot=350,
    max_wavelength_plot=1100,
    min_absorbance_plot=0.0, # New argument for y-axis min
    max_absorbance_plot=1.0  # New argument for y-axis max
):
    # Validate and set lambda_max
    try:
        lambda_max = int(lambda_max_input) if lambda_max_input else 790
    except ValueError:
        # Removed print statements to avoid clutter in the output area during interactive updates
        lambda_max = 790

    # Validate and set max_absorbance
    try:
        max_absorbance = float(max_absorbance_input) if max_absorbance_input else 1.0
        if max_absorbance <= 0: # Absorbance cannot be negative or zero in this context
            # Removed print statements
            max_absorbance = 1.0
    except ValueError:
        # Removed print statements
        max_absorbance = 1.0

    # Validate plot limits
    if not isinstance(min_wavelength_plot, (int, float)) or not isinstance(max_wavelength_plot, (int, float)):
        # Removed print statements
        min_wavelength_plot = 350
        max_wavelength_plot = 1100
    if min_wavelength_plot >= max_wavelength_plot:
        # Removed print statements
        min_wavelength_plot = 350
        max_wavelength_plot = 1100

    # Validate y-axis plot limits
    if not isinstance(min_absorbance_plot, (int, float)) or not isinstance(max_absorbance_plot, (int, float)):
        # Removed print statements
        min_absorbance_plot = 0.0
        max_absorbance_plot = 1.0
    if min_absorbance_plot >= max_absorbance_plot:
        # Removed print statements
        min_absorbance_plot = 0.0
        max_absorbance_plot = 1.0

    # Ensure plot limits are within the data generation range
    min_wavelength_plot = max(min_wavelength_plot, min(wavelengths_data))
    max_wavelength_plot = min(max_wavelength_plot, max(wavelengths_data))

    # Generate a synthetic absorbance spectrum using a Gaussian-like distribution
    # This simulates a broad absorption band based on the provided lambda_max and max_absorbance
    bandwidth = 70 # Adjust to control the breadth of the peak
    absorbance_data = max_absorbance * np.exp(-(wavelengths_data - lambda_max)**2 / (2 * bandwidth**2))

    # Add a small baseline absorbance
    absorbance_data += 0.05 * max_absorbance # Scale baseline with max_absorbance

    # Create the plot
    plt.figure(figsize=(10, 6))
    plt.plot(wavelengths_data, absorbance_data, color='blue')
    plt.title('Simulated Absorbance Spectrum of Copper Sulfate')
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Absorbance')

    # Mark lambda max on the plot
    plt.axvline(x=lambda_max, color='red', linestyle='--', label=rf'$\lambda_{{max}}$ = {lambda_max} nm')
    # Only add text if it's within the visible y-axis range to avoid clutter
    if max(absorbance_data) * 0.8 > min_absorbance_plot and max(absorbance_data) * 0.8 < max_absorbance_plot:
        plt.text(lambda_max + 10, max(absorbance_data) * 0.8, rf'$\lambda_{{max}}$ = {lambda_max} nm', color='red')

    # Apply the user-defined x-axis and y-axis limits
    plt.xlim(min_wavelength_plot, max_wavelength_plot)
    plt.ylim(min_absorbance_plot, max_absorbance_plot)

    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    # plt.show() # Removed plt.show() as it can interfere with interactive_output

# Create interactive widgets
lambda_max_widget = widgets.Text(
    value='',
    placeholder='790',
    description='Lambda Max (nm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

max_absorbance_widget = widgets.FloatText(
    value=1.0,
    min=0.01, # Molar absorptivity should be positive
    description='Molar Absorptivity (Arbitrary Units):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

min_wavelength_widget = widgets.IntText(
    value=350,
    min=200,
    description='Min Wavelength (nm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

max_wavelength_widget = widgets.IntText(
    value=1100,
    max=1200,
    description='Max Wavelength (nm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

min_absorbance_widget = widgets.FloatText(
    value=0.0,
    description='Min Absorbance:',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

max_absorbance_widget_y = widgets.FloatText(
    value=1.0,
    description='Max Absorbance:',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

# Group widgets into sections
curve_shape_widgets = widgets.VBox([
    widgets.HTML('<b>Curve Shape Parameters</b>'),
    lambda_max_widget,
    max_absorbance_widget
])

plot_display_widgets = widgets.VBox([
    widgets.HTML('<b>Plot Display Parameters</b>'),
    min_wavelength_widget,
    max_wavelength_widget,
    min_absorbance_widget,
    max_absorbance_widget_y
])

# Combine the control groups into a horizontal box
ui = widgets.HBox([curve_shape_widgets, plot_display_widgets])

# Use widgets.interactive_output to link the widgets to the plotting function
# This function returns an Output widget that will contain the plot.
plot_output = widgets.interactive_output(
    plot_copper_sulfate_spectrum,
    {
        'lambda_max_input': lambda_max_widget,
        'max_absorbance_input': max_absorbance_widget,
        'min_wavelength_plot': min_wavelength_widget,
        'max_wavelength_plot': max_wavelength_widget,
        'min_absorbance_plot': min_absorbance_widget,
        'max_absorbance_plot': max_absorbance_widget_y # Key matches function argument name
    }
)

# Display the UI (controls) and then the Output widget (plot)
display(ui, plot_output)

Output()